# JAX-MD: Propeller Twist Optimization

This notebook demonstrates gradient-based optimization of DNA force-field parameters to match a target propeller twist, using JAX-MD as the differentiable simulation backend.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

This notebook optimizes stacking parameters so the simulated propeller twist matches a target value. Because JAX-MD is fully differentiable, we can compute exact gradients through the simulation and use standard optimizers like Adam.

## Imports

In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp
import jax_md
import mythos.energy.dna1 as dna1_energy
import mythos.input.topology as jdna_top
import mythos.input.trajectory as jdna_traj
import mythos.losses.observable_wrappers as jdna_losses
import mythos.observables as jdna_obs
import mythos.simulators.jax_md as jdna_jaxmd
import optax
from mythos.energy.dna1.stacking import Stacking

jax.config.update("jax_enable_x64", True)

## Configuration

In [ ]:
N_SIM_STEPS = 20_000
N_OPT_STEPS = 25
LEARNING_RATE = 0.001
EXPERIMENT_DIR = Path("../../data/templates/simple-helix")

## Load topology and initial positions

In [ ]:
topology = jdna_top.from_oxdna_file(EXPERIMENT_DIR / "sys.top")
initial_positions = (
    jdna_traj.from_file(
        EXPERIMENT_DIR / "init.conf",
        topology.strand_counts,
        is_5p_3p=False,
    )
    .states[0]
    .to_rigid_body()
)

## Build the energy function and simulator

We use `create_default_energy_fn` to build the composed energy function, then mark `ss_stack_weights` as non-optimizable. We only optimize parameters from the `Stacking` energy term.

In [ ]:
energy_fn = dna1_energy.create_default_energy_fn(
    topology=topology,
    displacement_fn=jax_md.space.free()[0],
).with_noopt("ss_stack_weights")

transform_fn = dna1_energy.default_transform_fn()

# Only optimize parameters associated with the Stacking energy function
params = energy_fn.opt_params(from_fns=[Stacking])

experiment_config, _ = dna1_energy.default_configs()
dt = experiment_config["dt"]
kT = experiment_config["kT"]
diff_coef = experiment_config["diff_coef"]
rot_diff_coef = experiment_config["rot_diff_coef"]

gamma = jax_md.rigid_body.RigidBody(
    center=jnp.array([kT / diff_coef], dtype=jnp.float64),
    orientation=jnp.array([kT / rot_diff_coef], dtype=jnp.float64),
)
mass = jax_md.rigid_body.RigidBody(
    center=jnp.array([experiment_config["nucleotide_mass"]], dtype=jnp.float64),
    orientation=jnp.array([experiment_config["moment_of_inertia"]], dtype=jnp.float64),
)

simulator = jdna_jaxmd.JaxMDSimulator(
    energy_fn=energy_fn,
    simulator_params=jdna_jaxmd.StaticSimulatorParams(
        seq=jnp.array(topology.seq),
        mass=mass,
        bonded_neighbors=topology.bonded_neighbors,
        checkpoint_every=500,
        dt=dt,
        kT=kT,
        gamma=gamma,
    ),
    space=jax_md.space.free(),
    simulator_init=jax_md.simulate.nvt_langevin,
    neighbors=jdna_jaxmd.NoNeighborList(unbonded_nbrs=topology.unbonded_neighbors),
)

## Define the loss function

We use `ObservableLossFn` to wrap the propeller twist observable with a root mean squared error loss. The loss compares the simulated propeller twist against the known oxDNA target values.

In [ ]:
loss_fn = jdna_losses.ObservableLossFn(
    observable=jdna_obs.propeller.PropellerTwist(
        rigid_body_transform_fn=transform_fn,
        h_bonded_base_pairs=jnp.array([[1, 14], [2, 13], [3, 12], [4, 11], [5, 10], [6, 9]]),
    ),
    loss_fn=jdna_losses.RootMeanSquaredError(),
    return_observable=True,
)

weights = jnp.ones(N_SIM_STEPS, dtype=jnp.float64) / N_SIM_STEPS
target_prop_twist = jnp.array(jdna_obs.propeller.TARGETS["oxDNA"], dtype=jnp.float64)

## Build the differentiable loss and gradient function

Since JAX-MD is fully differentiable, we can use `jax.value_and_grad` directly through the simulation to compute exact gradients.

In [ ]:
def graddable_loss(in_params, in_key):
    in_key, subkey = jax.random.split(in_key)
    sim_out = simulator.run(in_params, initial_positions, N_SIM_STEPS, subkey)
    trajectory = sim_out.observables[0]  # there is only one observable here, the trajectory
    loss, ptwist = loss_fn(trajectory, target_prop_twist, weights)
    return loss, (ptwist, in_key)


grad_fn = jax.jit(jax.value_and_grad(graddable_loss, has_aux=True))

## Run the optimization loop

We use a simple Adam optimization loop. At each step, we run a simulation, compute the loss and gradients, and update the parameters.

In [ ]:
key = jax.random.key(1234)
optimizer = optax.adam(learning_rate=LEARNING_RATE)
opt_state = optimizer.init(params)

for i in range(N_OPT_STEPS):
    (loss, (prop_twist, key)), grads = grad_fn(params, key)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    print(f"Step {i}: Loss: {loss}, Measured: {prop_twist} Target: {target_prop_twist}")